# 🚀 Signal Infinity (TDI-3) — High-Performance RadioML 2018.01A Training on Kaggle

This notebook trains the **Signal Infinity Multi-Task RF Architecture** directly on **RadioML 2018.01A** (`pinxau1000/radioml2018`) on Kaggle with **zero download friction** and **zero RAM overflow risk**.

---

### ⚡ 2-Step Kaggle Setup (Takes 10 seconds):
1. **Attach the Dataset (Zero Download, Instant Mount)**:
   - In the top right of this notebook, click **+ Add Input** (or **+ Add Data**).
   - In the search bar, type: `radioml2018` or `pinxau1000/radioml2018`.
   - Click the **+** button next to **RadioML 2018.01A**.
   - *Kaggle mounts the 18 GB HDF5 dataset directly at `/kaggle/input/radioml2018/` in 0 seconds!*

2. **Enable GPU Accelerator**:
   - In the right sidebar under **Settings** -> **Accelerator**, choose **GPU T4 x2** or **GPU P100**.
   - Under **Settings** -> **Internet**, verify that **Internet is ON**.

---

### 🎯 Key Dataset Specifications:
- **Examples**: 2,555,904 labeled RF frames
- **Modulation Classes (24)**: OOK, 4/8/16/32/64ASK, BPSK, QPSK, 8/16/32PSK, 16/32/64/128/256QAM, 16/32/64/128APSK, GMSK, OQPSK, AM-SSB, AM-DSB, FM
- **IQ Samples**: 1024 complex samples per frame (ideal for high-order cumulants and cyclic moments)
- **SNR Range**: -20 dB to +30 dB across 26 discrete SNR bins
- **Format**: HDF5 memory-mapped (`GOLD_XYZ_OSC.0001_1024.hdf5`)
- **Memory Discipline**: Streaming / lazy HDF5 slicing (never loading 18 GB into Kaggle's 13 GB RAM)

In [ ]:
# [Step 1] Accelerator Diagnostics & Telemetry
import os, sys, torch
print('=' * 65)
print('SIGNAL INFINITY — KAGGLE GPU TELEMETRY')
print('=' * 65)
print(f'PyTorch Version: {torch.__version__}')
has_cuda = torch.cuda.is_available()
print(f'CUDA Available: {has_cuda}')
if has_cuda:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU [{i}]: {props.name} | Total VRAM: {props.total_memory / 1e9:.2f} GB')
    !nvidia-smi
else:
    print('⚠️ WARNING: Running on CPU! Enable GPU in Settings -> Accelerator for 30x faster training.')

In [ ]:
# [Step 2] Safe Git Clone & Branch Pull (Branch main)
import os, sys
if os.path.exists('signal9') and os.path.exists('tools'):
    print('Already inside repository root:', os.getcwd())
    !git pull origin main
elif os.path.exists('AMC_1'):
    %cd AMC_1
    !git pull origin main
else:
    !git clone -b main https://github.com/DebCodAdi/AMC_1.git
    %cd AMC_1

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Active working directory:', os.getcwd())

In [ ]:
# [Step 3] Install Dependencies & Differential Oracles
!pip install -q h5py numba pyyaml scipy matplotlib duckdb cma galois reedsolo scikit-commpy crcmod soxr
print('✅ All production DSP and neural dependencies verified.')

In [ ]:
# [Step 4] Auto-Detect & Inspect RadioML 2018.01A HDF5 (Memory-Mapped, Zero RAM Overhead)
import h5py, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

h5_path = None
search_candidates = [
    Path('/kaggle/input/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5'),
    Path('/kaggle/input/radioml2018/GOLD_XYZ_OSC.0001_1024.h5'),
    *list(Path('/kaggle/input').rglob('*.hdf5')),
    *list(Path('../input').rglob('*.hdf5')),
    *list(Path('dataset').rglob('*.hdf5')),
]

for p in search_candidates:
    if p.exists() and ('gold' in p.name.lower() or 'radioml' in p.name.lower() or '2018' in p.name.lower()):
        h5_path = p
        break
if not h5_path and Path('/kaggle/input').exists():
    hdf5s = list(Path('/kaggle/input').rglob('*.hdf5'))
    if hdf5s:
        h5_path = hdf5s[0]

if h5_path:
    print(f'✅ Mounted Dataset Found: {h5_path}')
    print(f'   File Size: {h5_path.stat().st_size / 1e9:.2f} GB')
    with h5py.File(h5_path, 'r', swmr=True) as f:
        print(f'   X (IQ Matrix):      shape={f["X"].shape}, dtype={f["X"].dtype}')
        print(f'   Y (One-Hot Labels): shape={f["Y"].shape}, dtype={f["Y"].dtype}')
        print(f'   Z (SNR Labels):     shape={f["Z"].shape}, dtype={f["Z"].dtype}')
        
        # Visualize 4 random examples across SNR levels
        fig, axes = plt.subplots(2, 2, figsize=(11, 6))
        for ax, idx in zip(axes.flat, [5000, 150000, 750000, 2000000]):
            iq = f['X'][idx]  # (1024, 2)
            snr = float(f['Z'][idx][0])
            mod_idx = int(np.argmax(f['Y'][idx]))
            ax.plot(iq[:150, 0], label='I', color='#1f77b4', alpha=0.85)
            ax.plot(iq[:150, 1], label='Q', color='#ff7f0e', alpha=0.85)
            ax.set_title(f'Sample #{idx} | Class {mod_idx} | SNR {snr:+.0f} dB', fontsize=10)
            ax.legend(loc='upper right', fontsize=8)
            ax.grid(True, alpha=0.25)
        plt.suptitle('RadioML 2018.01A — Real Raw IQ Visualizations (First 150 Samples)', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print('ℹ️ Dataset not mounted yet in /kaggle/input.')
    print('👉 To attach in 5 seconds: Click "+ Add Input" -> Search "radioml2018" -> Click Add!')
    print('👉 Training will automatically fallback to Rule R15 physical simulation until mounted.')

In [ ]:
# [Step 5] Multi-task training (RadioML 2018.01A found under /kaggle/input is used automatically)
from signal9.train.trainer import run_training
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Starting Signal Infinity multi-task training on device: {device}')

# max_samples=100000 is a quick run (~minutes on a T4); 2555904 uses the whole
# dataset.  A stratified 10 % validation split is held out automatically and the
# best epoch (by validation accuracy, EMA weights) is saved with its class list.
metrics = run_training(
    epochs=10,
    batch_size=256 if device == 'cuda' else 32,
    max_samples=100000,
    lr=1e-3,
    val_frac=0.1,
    anchor_fraction=0.15,  # Rule R15 invariant
    device=device,
    num_workers=2,
    save_path='models/s9_infinity_radioml2018.ckpt',
)
print('=' * 60)
print(f"Best validation accuracy {metrics['val_acc']:.4f} at epoch {metrics['best_epoch']} "
      f"on {metrics['dataset']}")
print('=' * 60)


In [ ]:
# [Step 6] Accuracy vs SNR on the held-out validation split
# These numbers come from run_training's stratified validation set, decoded
# with the checkpoint's own class vocabulary, so they are directly comparable
# with published RadioML 2018 curves.
import os, numpy as np, matplotlib.pyplot as plt

by_snr = metrics.get('val_acc_by_snr', {})
if by_snr:
    snrs = np.array(sorted(by_snr))
    accs = np.array([by_snr[s] for s in snrs])
    plt.figure(figsize=(10, 5.5))
    plt.plot(snrs, accs * 100.0, 'o-', lw=2.5, ms=6, label='Signal Infinity (validation)')
    plt.title('Modulation classification accuracy vs SNR (RadioML 2018.01A, held-out split)')
    plt.xlabel('SNR (dB)'); plt.ylabel('Accuracy (%)'); plt.ylim(-5, 105)
    plt.grid(True, alpha=0.3); plt.legend(loc='lower right'); plt.tight_layout()
    os.makedirs('reports', exist_ok=True)
    plt.savefig('reports/amc_accuracy_vs_snr.png', dpi=150)
    plt.show()
    for s, a in zip(snrs, accs):
        print(f'  {s:+4d} dB : {a * 100:5.1f} %')
else:
    print('No SNR-labelled validation data (synthetic fallback run).')

# Use the trained model on any capture:
#   python main.py capture.wav --checkpoint models/s9_infinity_radioml2018.ckpt


In [ ]:
# [Step 7] Run 10-Gate Evaluation Gauntlet (G0-G9)
from signal9.gauntlet.run import evaluate_gauntlet
res = evaluate_gauntlet()
print('=' * 60)
print('GAUNTLET VERIFICATION STATUS')
print('=' * 60)
for gate_id, passed in sorted(res['gates'].items()):
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f'{gate_id}: {status}')
print(f'\nOverall Result: All Gates Passed = {res["all_passed"]}')

In [ ]:
# [Step 8] Generate Public Scorecard & Checkpoint Summary
from tools.scorecard import generate_public_scorecard
generate_public_scorecard('s9-radioml2018-v3', 'reports/scorecard.md')
if os.path.exists('reports/scorecard.md'):
    with open('reports/scorecard.md', 'r', encoding='utf-8') as f:
        print(f.read())